# Neural Ledger — Semantic Recall

Neural Ledger retrieves memories by meaning, not just by matching words.

This notebook shows one concrete consequence: a query worded differently from a stored memory still surfaces the right result — because semantic recall compares intent, not text.

**Arc:** `store → keyword baseline → semantic recall → side-by-side → feedback reinforcement`

## Setup

In [1]:
import sys
sys.path.insert(0, str(__import__("pathlib").Path("..").resolve()))

from neural_ledger import Memory

# Neural Ledger picks up sentence-transformers automatically when installed.
# Run:  pip install sentence-transformers
mem = Memory()
print("Semantic available:", mem._runtime.semantic.available)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic available: True


## Records

Ten memories covering API errors, auth issues, and noise.
Two are genuinely relevant to auth/token problems; three are lexically similar distractors; five are noise.

In [2]:
records_raw = [
    # Relevant
    {"id": "r1", "content": "GitHub API request failed with 401 because the access token had expired.", "kind": "observation"},
    {"id": "r2", "content": "Refreshing the GitHub token and retrying resolved the 401 error.", "kind": "procedure"},
    # Lexically similar distractors
    {"id": "r3", "content": "GitHub API rate limit caused a temporary 403 response.", "kind": "observation"},
    {"id": "r4", "content": "GitHub pull request fetch worked after widening repository permissions.", "kind": "observation"},
    {"id": "r5", "content": "Slack webhook failed because the signing secret was missing.", "kind": "observation"},
    # Noise
    {"id": "r6", "content": "Use terse bullet points in status updates.", "kind": "note"},
    {"id": "r7", "content": "Database connection failed due to an incorrect host value.", "kind": "observation"},
    {"id": "r8", "content": "Retry with exponential backoff for transient upstream failures.", "kind": "note"},
    {"id": "r9", "content": "The CI pipeline failed because the package lockfile was out of date.", "kind": "observation"},
    {"id": "r10", "content": "Check whether environment variables are loaded before retrying.", "kind": "procedure"},
]

record_map = {}
for r in records_raw:
    rec = mem.remember(r["content"], kind=r["kind"])
    record_map[rec.id] = r["id"]

rev_map = {v: k for k, v in record_map.items()}

print(f"Stored {len(records_raw)} records.\n")
for r in records_raw:
    print(f"  [{r['id']}] ({r['kind']}) {r['content']}")

Stored 10 records.

  [r1] (observation) GitHub API request failed with 401 because the access token had expired.
  [r2] (procedure) Refreshing the GitHub token and retrying resolved the 401 error.
  [r3] (observation) GitHub API rate limit caused a temporary 403 response.
  [r4] (observation) GitHub pull request fetch worked after widening repository permissions.
  [r5] (observation) Slack webhook failed because the signing secret was missing.
  [r6] (note) Use terse bullet points in status updates.
  [r7] (observation) Database connection failed due to an incorrect host value.
  [r8] (note) Retry with exponential backoff for transient upstream failures.
  [r9] (observation) The CI pipeline failed because the package lockfile was out of date.
  [r10] (procedure) Check whether environment variables are loaded before retrying.


## Baseline — keyword recall

The query uses none of the words "token", "expired", or "401".
Keyword matching has little signal; r1 ends up at rank 3.

In [3]:
QUERY = "Have we seen auth problems with outbound API calls before?"

def show(hits):
    print(f"  {'Rank':<5} {'ID':<5} {'Score':<8} Content")
    print(f"  {'-'*75}")
    for i, h in enumerate(hits[:5]):
        sid = record_map.get(h.id, "?")
        print(f"  {i+1:<5} {sid:<5} {h.score:<8.4f} {h.content[:60]}")
    print()

from neural_ledger.retrieve.semantic import SemanticRetriever

_saved = mem._runtime.semantic
mem._runtime.semantic = SemanticRetriever(encoder=None)  # keyword-only

hits_keyword = mem.recall(QUERY, limit=5)

mem._runtime.semantic = _saved  # restore

print(f'Query: "{QUERY}"\n')
print("Keyword baseline:")
show(hits_keyword)

Query: "Have we seen auth problems with outbound API calls before?"

Keyword baseline:
  Rank  ID    Score    Content
  ---------------------------------------------------------------------------
  1     r10   0.2738   Check whether environment variables are loaded before retryi
  2     r3    0.2729   GitHub API rate limit caused a temporary 403 response.
  3     r1    0.2720   GitHub API request failed with 401 because the access token 
  4     r9    0.2000   The CI pipeline failed because the package lockfile was out 
  5     r8    0.2000   Retry with exponential backoff for transient upstream failur



## Semantic recall

With `sentence-transformers`, both the query and each record are encoded as dense vectors and ranked by cosine similarity. "Auth problems with outbound API calls" maps semantically to "access token had expired" — the right record rises to rank 1.

In [4]:
hits_semantic = mem.recall(QUERY, with_why=True, limit=5)

print(f'Query: "{QUERY}"\n')
print("Semantic recall:")
show(hits_semantic)

top = hits_semantic[0]
print(f"Why top hit: {top.why}")

Query: "Have we seen auth problems with outbound API calls before?"

Semantic recall:
  Rank  ID    Score    Content
  ---------------------------------------------------------------------------
  1     r1    0.5786   GitHub API request failed with 401 because the access token 
  2     r3    0.5455   GitHub API rate limit caused a temporary 403 response.
  3     r9    0.4471   The CI pipeline failed because the package lockfile was out 
  4     r8    0.4400   Retry with exponential backoff for transient upstream failur
  5     r2    0.3914   Refreshing the GitHub token and retrying resolved the 401 er

Why top hit: Retrieved by semantic similarity (seed score 0.38). Expanded through 2 related records via learned links. This memory is recent and active.


## Comparison

In [5]:
print(f'Query: "{QUERY}"\n')
print(f"  {'Rank':<5} {'Keyword':<38} Semantic")
print(f"  {'-'*80}")
for i in range(5):
    kw  = record_map.get(hits_keyword[i].id, "-")  if i < len(hits_keyword)  else "-"
    sm  = record_map.get(hits_semantic[i].id, "-") if i < len(hits_semantic) else "-"
    kw_txt = hits_keyword[i].content[:32]  if i < len(hits_keyword)  else ""
    sm_txt = hits_semantic[i].content[:32] if i < len(hits_semantic) else ""
    print(f"  {i+1:<5} [{kw}] {kw_txt:<32}  [{sm}] {sm_txt}")

kw_r1  = next((i+1 for i, h in enumerate(hits_keyword)  if record_map.get(h.id) == "r1"), "absent")
sem_r1 = next((i+1 for i, h in enumerate(hits_semantic) if record_map.get(h.id) == "r1"), "absent")
print()
print(f"r1 (token-expiry cause) — keyword rank: {kw_r1}  →  semantic rank: {sem_r1}")
print("The query shares no words with 'token', 'expired', or '401'.")

Query: "Have we seen auth problems with outbound API calls before?"

  Rank  Keyword                                Semantic
  --------------------------------------------------------------------------------
  1     [r10] Check whether environment variab  [r1] GitHub API request failed with 4
  2     [r3] GitHub API rate limit caused a t  [r3] GitHub API rate limit caused a t
  3     [r1] GitHub API request failed with 4  [r9] The CI pipeline failed because t
  4     [r9] The CI pipeline failed because t  [r8] Retry with exponential backoff f
  5     [r8] Retry with exponential backoff f  [r2] Refreshing the GitHub token and 

r1 (token-expiry cause) — keyword rank: 3  →  semantic rank: 1
The query shares no words with 'token', 'expired', or '401'.


## Extension — feedback reinforcement

Positive feedback on the correct hit raises its usefulness prior. On the next recall it scores higher still.

In [6]:
r1_hit = next(h for h in hits_semantic if record_map.get(h.id) == "r1")
print(f'Applying positive feedback to [r1]: "{r1_hit.content}"\n')

mem.feedback([r1_hit], helped=True, reason="Correct — the token had expired.")

hits_after = mem.recall(QUERY, limit=5)

print("After feedback:")
show(hits_after)

before_score = r1_hit.score
after_r1 = next(h for h in hits_after if h.id == r1_hit.id)
print(f"[r1] score: {before_score:.4f} → {after_r1.score:.4f} after one feedback signal.")

Applying positive feedback to [r1]: "GitHub API request failed with 401 because the access token had expired."

After feedback:
  Rank  ID    Score    Content
  ---------------------------------------------------------------------------
  1     r1    0.6012   GitHub API request failed with 401 because the access token 
  2     r3    0.5533   GitHub API rate limit caused a temporary 403 response.
  3     r2    0.5515   Refreshing the GitHub token and retrying resolved the 401 er
  4     r9    0.4549   The CI pipeline failed because the package lockfile was out 
  5     r8    0.4477   Retry with exponential backoff for transient upstream failur

[r1] score: 0.5786 → 0.6012 after one feedback signal.


## Summary

Semantic recall surfaced the right memory when the query contained none of the stored record's key words. Feedback then raised its score further, reinforcing the result for future recalls.

**Semantic recall finds the right memory when the wording changes; feedback then helps the engine learn that this memory is useful.**